# 05 — Gold: FactSales (Transacional)

Grain: uma linha por `(OrderID, ProductID)` de `bronze.order_details`.

**Técnica:** `INSERT OR REPLACE INTO` com UNIQUE em SalesSK — idempotente.

**Joins com SCD2:**
- `DimCustomer`: versão vigente na data do pedido (`ValidFrom <= OrderDate < ValidTo`)
- `DimProduct`: idem

**Métricas:**
- `GrossRevenue = UnitPrice × Quantity`
- `NetRevenue   = UnitPrice × Quantity × (1 - Discount)`

**Diferencial DuckDB:** `QUALIFY` para filtrar diretamente no SELECT sem subquery externa.

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())
from utils import get_conn, DB_PATH, DATA_DIR

conn = get_conn()
print(f"Conectado: {DB_PATH}")

Conectado: /workspace/pf_northwind/duckdb/northwind_dw.duckdb


In [2]:
# ============================================================
# Preview da fonte: order_details + orders
# ============================================================
n_od = conn.execute("SELECT COUNT(*) FROM bronze.order_details").fetchone()[0]
print(f"bronze.order_details: {n_od} linhas (grain da FactSales)")

print("\nPreview order_details + orders:")
print(conn.execute("""
    SELECT od.OrderID, od.ProductID, o.CustomerID, o.OrderDate::DATE AS OrderDate,
           od.UnitPrice, od.Quantity, od.Discount,
           od.UnitPrice * od.Quantity                         AS GrossRevenue,
           od.UnitPrice * od.Quantity * (1 - od.Discount)     AS NetRevenue
    FROM bronze.order_details od
    JOIN bronze.orders o ON od.OrderID = o.OrderID
    LIMIT 5
""").fetchdf().to_string(index=False))

bronze.order_details: 2155 linhas (grain da FactSales)

Preview order_details + orders:
 OrderID  ProductID CustomerID  OrderDate  UnitPrice  Quantity  Discount  GrossRevenue  NetRevenue
   10248         11      VINET 1996-07-04       14.0        12       0.0         168.0       168.0
   10248         42      VINET 1996-07-04        9.8        10       0.0          98.0        98.0
   10248         72      VINET 1996-07-04       34.8         5       0.0         174.0       174.0
   10249         14      TOMSP 1996-07-05       18.6         9       0.0         167.4       167.4
   10249         51      TOMSP 1996-07-05       42.4        40       0.0        1696.0      1696.0


In [3]:
# ============================================================
# FactSales — INSERT OR REPLACE (idempotente via SalesSK UNIQUE)
# JOIN com SCD2: versão vigente na data do pedido
# ============================================================
conn.execute("""
    INSERT OR REPLACE INTO gold.FactSales
    SELECT
        CAST(hash(CAST(od.OrderID AS VARCHAR) || ',' || CAST(od.ProductID AS VARCHAR)) % 2147483647 AS INTEGER) AS SalesSK,
        CAST(strftime(o.OrderDate::DATE, '%Y%m%d') AS INTEGER)  AS OrderDateKey,
        dc.CustomerSK,
        dp.ProductSK,
        de.EmployeeSK,
        ds.ShipperSK,
        od.OrderID,
        od.ProductID,
        od.UnitPrice,
        od.Quantity,
        od.Discount,
        od.UnitPrice * od.Quantity                          AS GrossRevenue,
        od.UnitPrice * od.Quantity * (1.0 - od.Discount)   AS NetRevenue,
        current_timestamp                                   AS LoadTimestamp
    FROM bronze.order_details od
    JOIN bronze.orders o ON od.OrderID = o.OrderID
    -- DimCustomer: versão vigente na data do pedido
    JOIN gold.DimCustomer dc
        ON o.CustomerID = dc.CustomerID
        AND o.OrderDate::DATE >= dc.ValidFrom
        AND o.OrderDate::DATE <  dc.ValidTo
    -- DimProduct: versão vigente na data do pedido
    JOIN gold.DimProduct dp
        ON od.ProductID = dp.ProductID
        AND o.OrderDate::DATE >= dp.ValidFrom
        AND o.OrderDate::DATE <  dp.ValidTo
    JOIN gold.DimEmployee de ON o.EmployeeID = de.EmployeeID
    JOIN gold.DimShipper  ds ON o.ShipVia    = ds.ShipperID
""")

n = conn.execute("SELECT COUNT(*) FROM gold.FactSales").fetchone()[0]
print(f"FactSales após INSERT OR REPLACE: {n} linhas (esperado: 2155)")

FactSales após INSERT OR REPLACE: 2155 linhas (esperado: 2155)


In [4]:
# ============================================================
# Validações
# ============================================================
print("1. Contagem total:")
n_fact   = conn.execute("SELECT COUNT(*) FROM gold.FactSales").fetchone()[0]
n_bronze = conn.execute("SELECT COUNT(*) FROM bronze.order_details").fetchone()[0]
print(f"   FactSales={n_fact}, bronze.order_details={n_bronze}, Match={n_fact == n_bronze}")

print("\n2. Receita total:")
print(conn.execute("""
    SELECT
        ROUND(SUM(GrossRevenue), 2) AS GrossRevenue,
        ROUND(SUM(NetRevenue),   2) AS NetRevenue,
        ROUND(SUM(GrossRevenue - NetRevenue), 2) AS Desconto
    FROM gold.FactSales
""").fetchdf().to_string(index=False))

print("\n3. Órfãos (sem dimensão correspondente):")
o_cust = conn.execute("""
    SELECT COUNT(*) FROM gold.FactSales f
    WHERE NOT EXISTS (SELECT 1 FROM gold.DimCustomer c WHERE c.CustomerSK = f.CustomerSK)
""").fetchone()[0]
o_prod = conn.execute("""
    SELECT COUNT(*) FROM gold.FactSales f
    WHERE NOT EXISTS (SELECT 1 FROM gold.DimProduct p WHERE p.ProductSK = f.ProductSK)
""").fetchone()[0]
print(f"   Órfãos por CustomerSK: {o_cust} (esperado: 0)")
print(f"   Órfãos por ProductSK:  {o_prod} (esperado: 0)")

print("\n4. Top 5 produtos por NetRevenue:")
print(conn.execute("""
    SELECT p.ProductName, p.CategoryName,
           ROUND(SUM(f.NetRevenue), 2) AS NetRevenue
    FROM gold.FactSales f
    JOIN gold.DimProduct p ON f.ProductSK = p.ProductSK
    GROUP BY p.ProductName, p.CategoryName
    ORDER BY NetRevenue DESC
    LIMIT 5
""").fetchdf().to_string(index=False))

print("\n5. QUALIFY — top 1 produto por categoria (técnica DuckDB):")
print(conn.execute("""
    SELECT p.CategoryName, p.ProductName, ROUND(SUM(f.NetRevenue), 2) AS NetRevenue
    FROM gold.FactSales f
    JOIN gold.DimProduct p ON f.ProductSK = p.ProductSK
    GROUP BY p.CategoryName, p.ProductName
    QUALIFY RANK() OVER (PARTITION BY p.CategoryName ORDER BY SUM(f.NetRevenue) DESC) = 1
    ORDER BY NetRevenue DESC
""").fetchdf().to_string(index=False))

conn.close()

1. Contagem total:
   FactSales=2155, bronze.order_details=2155, Match=True

2. Receita total:
 GrossRevenue  NetRevenue  Desconto
   1354458.59  1265793.04  88665.55

3. Órfãos (sem dimensão correspondente):
   Órfãos por CustomerSK: 0 (esperado: 0)
   Órfãos por ProductSK:  0 (esperado: 0)

4. Top 5 produtos por NetRevenue:
            ProductName   CategoryName  NetRevenue
          Côte de Blaye      Beverages   141396.73
Thüringer Rostbratwurst   Meat/Poultry    80368.67
   Raclette Courdavault Dairy Products    71155.70
         Tarte au sucre    Confections    47234.97
      Camembert Pierrot Dairy Products    46825.48

5. QUALIFY — top 1 produto por categoria (técnica DuckDB):
  CategoryName             ProductName  NetRevenue
     Beverages           Côte de Blaye   141396.73
  Meat/Poultry Thüringer Rostbratwurst    80368.67
Dairy Products    Raclette Courdavault    71155.70
   Confections          Tarte au sucre    47234.97
Grains/Cereals  Gnocchi di nonna Alice    42593.06
